# NDI v1.4 pre-checks (read-only): p12 cause, wellness zero counts

Run this **before** anything is changed or tagged (v1.4 memo, deliverables 1 and 2). Runtime > Run all, about 5 minutes.

- CELL 1.0, CELL 2 and CELL 5 are copied unchanged from `NDI_v1_3_forward.ipynb`, so CELL 5 prints the same `ERROR p12` line as before.
- **P-1** confirms the cause of the p12 error: the `repr()` of p12's max heart rate cell, the CELL 5 expression on it, and `load_pmdata_participant` on p12's folder outside any `try` block (deliverable 1).
- **W-0** counts zeros per participant and 1-5 item, the reports W3 drops, and the W1 and W4 counts (deliverable 2). Same functions as W-1 in `NDI_v1_4.ipynb`.
- **S-0** (FYI, not in the memo) counts sleep scores whose local date differs from the date CELL 5 gives them.

Writes only to `results_diagnostics/` (p12_check.txt, wellness_zero_counts.csv, wellness_rules_precheck_log.csv, sleep_score_dating_fyi.csv). Nothing in any results folder is touched.

In [1]:
# ================================================================
# CELL 1.0  CONFIGURATION  (edit paths to match your Drive)
# Brief Section 6: all parameters in one place, fixed seed
# ================================================================

#  Google Drive paths
PIPELINE_MODULE   = "/content/drive/MyDrive/ndi_project/ndi_pipeline.py"
DEPRESJON_DIR     = "/content/drive/MyDrive/ndi_project/depresjon-dataset/data"
PMDATA_DIR        = "/content/drive/MyDrive/ndi_project/pmdata"
PMDATA_OVERVIEW   = "/content/drive/MyDrive/ndi_project/pmdata/participant-overview.xlsx"
RESULTS_DIR       = "/content/drive/MyDrive/ndi_project/results"

#  Reproducibility seed (Brief Section 6)
RANDOM_SEED = 42

#  Mount Drive
from google.colab import drive
drive.mount("/content/drive")

import os, sys
os.makedirs(RESULTS_DIR, exist_ok=True)

# Add pipeline module to path
sys.path.insert(0, os.path.dirname(PIPELINE_MODULE))

print("Drive mounted.")
print("DEPRESJON dir exists:", os.path.isdir(DEPRESJON_DIR))
print("PMData    dir exists:", os.path.isdir(PMDATA_DIR))
print("Pipeline  exists    :", os.path.isfile(PIPELINE_MODULE))


Mounted at /content/drive
Drive mounted.
DEPRESJON dir exists: True
PMData    dir exists: True
Pipeline  exists    : True


In [2]:
# ================================================================
# CELL 2  INSTALL DEPENDENCIES + IMPORTS
# ================================================================
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["xgboost", "shap"]:
    install(pkg)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.tri as tri
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings("ignore")
np.random.seed(RANDOM_SEED)

import importlib.util, sys as _sys

spec = importlib.util.spec_from_file_location("ndi_pipeline", PIPELINE_MODULE)
ndi  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ndi)

print("ndi_pipeline loaded. Components available:")
for fn in ["neutrosophic_decompose","blocked_tscv","run_baselines",
           "lambda_grid_search","shap_attribution"]:
    print(f"  ndi.{fn}: {'OK' if hasattr(ndi, fn) else 'MISSING'}")


ndi_pipeline loaded. Components available:
  ndi.neutrosophic_decompose: OK
  ndi.blocked_tscv: OK
  ndi.run_baselines: OK
  ndi.lambda_grid_search: OK
  ndi.shap_attribution: OK


In [3]:
# ================================================================
# CELL 5 -- LOAD PMDATA DATASET
# Actual structure:
#   pmdata/p01/fitbit/heart_rate.json
#   pmdata/p01/fitbit/resting_heart_rate.json
#   pmdata/p01/fitbit/sleep.json
#   pmdata/p01/fitbit/sleep_score.csv
#   pmdata/p01/fitbit/steps.json
#   pmdata/p01/fitbit/lightly_active_minutes.json
#   pmdata/p01/fitbit/moderately_active_minutes.json
#   pmdata/p01/fitbit/very_active_minutes.json
#   pmdata/p01/fitbit/sedentary_minutes.json
#   pmdata/p01/pmsys/wellness.csv    <- daily mood/wellness self-report
#   pmdata/p01/pmsys/srpe.csv
#   pmdata/participant-overview.xlsx <- max HR, age, gender per participant
# ================================================================
import os, glob, json as _json
import pandas as pd
import numpy as np
import openpyxl  # for reading xlsx

#  Load participant overview
try:
    overview = pd.read_excel(PMDATA_OVERVIEW, header=1)
    overview.columns = [str(c).strip().lower().replace(" ","_")
                        for c in overview.columns]
    overview = overview.dropna(subset=[overview.columns[0]])
    print("participant-overview columns:", list(overview.columns))
    print(overview.head(3))
except Exception as e:
    print(f"Could not load participant-overview: {e}")
    overview = pd.DataFrame()

#  Helper: load one JSON Fitbit file
def load_fitbit_json(path):
    """Load a Fitbit JSON file. Returns list of {dateTime, value} dicts."""
    try:
        with open(path) as f:
            data = _json.load(f)
        if isinstance(data, list):
            return data
        return []
    except Exception:
        return []

def fitbit_to_daily(entries, value_key="value", agg="mean"):
    """Aggregate minute/entry-level Fitbit data to daily values."""
    rows = {}
    for entry in entries:
        date = str(entry.get("dateTime",""))[:10]
        val  = entry.get(value_key, entry.get("value", None))
        if date and val is not None:
            try:
                val = float(val)
                rows.setdefault(date, []).append(val)
            except (ValueError, TypeError):
                pass
    if agg == "mean":
        return {d: np.mean(v) for d,v in rows.items()}
    elif agg == "sum":
        return {d: np.sum(v) for d,v in rows.items()}
    elif agg == "max":
        return {d: np.max(v) for d,v in rows.items()}
    return rows

#  Helper: load resting heart rate JSON
def load_resting_hr(path):
    """resting_heart_rate.json has nested value.value structure."""
    entries = load_fitbit_json(path)
    rows = {}
    for entry in entries:
        date = str(entry.get("dateTime",""))[:10]
        val  = entry.get("value",{})
        if isinstance(val, dict):
            rhr = val.get("value", val.get("restingHeartRate", None))
        else:
            rhr = val
        if date and rhr is not None:
            try:
                rows[date] = float(rhr)
            except (ValueError, TypeError):
                pass
    return rows

#  Helper: load sleep.json
def load_sleep(path):
    """Extract sleep efficiency and minutes asleep from sleep.json."""
    entries = load_fitbit_json(path)
    eff_rows = {}; min_rows = {}
    for entry in entries:
        # Each entry is one sleep session
        date = str(entry.get("dateOfSleep", entry.get("dateTime","")))[:10]
        eff  = entry.get("efficiency", None)
        mins = entry.get("minutesAsleep", None)
        if date and eff is not None:
            try:
                eff_rows[date] = float(eff)
            except (ValueError, TypeError):
                pass
        if date and mins is not None:
            try:
                min_rows[date] = float(mins)
            except (ValueError, TypeError):
                pass
    return eff_rows, min_rows

#  Helper: load sleep_score.csv
def load_sleep_score(path):
    """sleep_score.csv: timestamp, overall_score, composition_score, ..."""
    try:
        df = pd.read_csv(path)
        df.columns = [c.strip().lower() for c in df.columns]
        # Timestamp column
        ts_col = next((c for c in df.columns if "timestamp" in c or "date" in c), None)
        score_col = next((c for c in df.columns if "overall" in c or "score" in c), None)
        if ts_col and score_col:
            df["date"] = pd.to_datetime(df[ts_col]).dt.strftime("%Y-%m-%d")
            return dict(zip(df["date"], pd.to_numeric(df[score_col], errors="coerce")))
    except Exception:
        pass
    return {}

#  Helper: load wellness.csv
def load_wellness(path):
    """
    pmsys/wellness.csv: daily self-report.
    Columns typically: date, fatigue, mood, readiness, sleep_duration,
                       sleep_quality, stress, soreness
    """
    try:
        df = pd.read_csv(path)
        df.columns = [c.strip().lower().replace(" ","_") for c in df.columns]
        if "date" not in df.columns:
            return pd.DataFrame()
        df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
        return df
    except Exception:
        return pd.DataFrame()

#  Helper: load googledocs/reporting.csv
def load_reporting(path):
    """
    reporting.csv: date, timestamp, meals, weight, glasses_of_fluid, alcohol
    """
    try:
        df = pd.read_csv(path)
        df.columns = [c.strip().lower().replace(" ","_") for c in df.columns]
        date_col = next((c for c in df.columns if "date" in c), None)
        if date_col:
            df["date"] = pd.to_datetime(df[date_col]).dt.strftime("%Y-%m-%d")
            return df
    except Exception:
        pass
    return pd.DataFrame()

#  Main loader
def load_pmdata_participant(pid_dir):
    """
    Load all data for one PMData participant.
    Returns a daily DataFrame with all available signals.
    """
    fitbit_dir    = os.path.join(pid_dir, "fitbit")
    pmsys_dir     = os.path.join(pid_dir, "pmsys")
    googledocs_dir= os.path.join(pid_dir, "googledocs")

    daily = {}  # date -> feature dict

    def add_series(series_dict, col_name):
        for date, val in series_dict.items():
            daily.setdefault(date, {})[col_name] = val

    # Resting heart rate (most reliable HR signal for daily level)
    rhr_path = os.path.join(fitbit_dir, "resting_heart_rate.json")
    if os.path.isfile(rhr_path):
        add_series(load_resting_hr(rhr_path), "resting_hr")

    # Steps
    steps_path = os.path.join(fitbit_dir, "steps.json")
    if os.path.isfile(steps_path):
        add_series(fitbit_to_daily(load_fitbit_json(steps_path), agg="sum"), "steps")

    # Activity minutes
    for fname, col in [
        ("lightly_active_minutes.json",    "lightly_active_min"),
        ("moderately_active_minutes.json", "moderately_active_min"),
        ("very_active_minutes.json",       "very_active_min"),
        ("sedentary_minutes.json",         "sedentary_min"),
    ]:
        p = os.path.join(fitbit_dir, fname)
        if os.path.isfile(p):
            add_series(fitbit_to_daily(load_fitbit_json(p), agg="sum"), col)

    # Sleep
    sleep_path = os.path.join(fitbit_dir, "sleep.json")
    if os.path.isfile(sleep_path):
        eff, mins = load_sleep(sleep_path)
        add_series(eff,  "sleep_efficiency")
        add_series(mins, "sleep_minutes")

    # Sleep score
    ss_path = os.path.join(fitbit_dir, "sleep_score.csv")
    if os.path.isfile(ss_path):
        add_series(load_sleep_score(ss_path), "sleep_score")

    if not daily:
        return pd.DataFrame()

    # Build base daily DataFrame
    df = pd.DataFrame.from_dict(daily, orient="index")
    df.index.name = "date"
    df = df.reset_index()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    # Active minutes total
    act_cols = [c for c in ["lightly_active_min","moderately_active_min","very_active_min"] if c in df.columns]
    if act_cols:
        df["active_minutes_total"] = df[act_cols].sum(axis=1)

    # Merge wellness
    w_path = os.path.join(pmsys_dir, "wellness.csv")
    if os.path.isfile(w_path):
        well = load_wellness(w_path)
        if not well.empty:
            well["date"] = pd.to_datetime(well["date"])
            df = df.merge(well, on="date", how="left")

    # Merge reporting (weight, fluid)
    r_path = os.path.join(googledocs_dir, "reporting.csv")
    if os.path.isfile(r_path):
        rep = load_reporting(r_path)
        if not rep.empty:
            rep["date"] = pd.to_datetime(rep["date"])
            rep_agg = rep.groupby("date").agg(
                weight         = ("weight", "mean"),
                glasses_of_fluid=("glasses_of_fluid","mean") if "glasses_of_fluid" in rep.columns else ("date","count"),
            ).reset_index()
            df = df.merge(rep_agg, on="date", how="left")

    df["date"] = df["date"].dt.strftime("%Y-%m-%d")
    return df


#  Load all participants
pmdata_participants = []
participant_dirs = sorted([
    os.path.join(PMDATA_DIR, d)
    for d in os.listdir(PMDATA_DIR)
    if os.path.isdir(os.path.join(PMDATA_DIR, d)) and d.startswith("p")
])
print(f"Found {len(participant_dirs)} participant directories")

for pdir in participant_dirs:
    pid = os.path.basename(pdir)
    try:
        daily = load_pmdata_participant(pdir)
        if len(daily) < 7:
            print(f"  SKIP {pid}: only {len(daily)} days")
            continue

        # Get max HR from overview
        max_hr = np.nan
        if not overview.empty:
            hr_col = next((c for c in overview.columns if "heart" in c and "max" in c), None)
            id_col = next((c for c in overview.columns if "participant" in c or "id" in c), None)
            if hr_col and id_col:
                row = overview[overview[id_col].astype(str).str.lower().str.strip() == pid.lower()]
                if not row.empty:
                    max_hr = float(row.iloc[0][hr_col]) if not pd.isna(row.iloc[0][hr_col]) else np.nan

        pmdata_participants.append(dict(
            pid       = pid,
            group     = "pmdata",
            is_depressed = 0,
            max_hr    = max_hr,
            n_days    = len(daily),
            days_df   = daily,
        ))
        cols_found = [c for c in daily.columns if c != "date"]
        print(f"  OK {pid}: {len(daily)} days  max_HR={max_hr:.0f}  signals={cols_found[:6]}")
    except Exception as e:
        print(f"  ERROR {pid}: {e}")

print(f"\nPMData: {len(pmdata_participants)} participants loaded")
if pmdata_participants:
    print(f"  Median days: {np.median([p['n_days'] for p in pmdata_participants]):.0f}")
    all_cols = set()
    for p in pmdata_participants:
        all_cols.update(p['days_df'].columns.tolist())
    print(f"  All available columns: {sorted(all_cols)}")


participant-overview columns: ['participant_id', 'age', 'height', 'gender', 'a_or_b_person', 'max_heart_rate', 'date', 'minutes', 'seconds', 'stride_walk', 'stride_run']
  participant_id  age  height gender a_or_b_person max_heart_rate  \
0            p01   48     195   male             A            182   
1            p02   60     180   male             A            169   
2            p03   25     184   male             A            157   

                  date minutes seconds  stride_walk  stride_run  
0  2019-11-26 00:00:00      29      33         80.9       102.9  
1  2019-12-15 00:00:00      23      51         74.7        92.4  
2  2019-12-30 00:00:00      33      22          NaN         NaN  
Found 16 participant directories
  OK p01: 152 days  max_HR=182  signals=['resting_hr', 'steps', 'lightly_active_min', 'moderately_active_min', 'very_active_min', 'sedentary_min']
  OK p02: 152 days  max_HR=169  signals=['resting_hr', 'steps', 'lightly_active_min', 'moderately_active_min'

In [4]:
# P-1  p12: confirm the cause of the loader error before changing anything (v1.4 memo, Section 2; deliverable 1)
# Read-only. Uses CELL 5 exactly as it ran in v1.3 (unchanged above). Writes results_diagnostics/p12_check.txt only.
import os, io, contextlib, traceback
import numpy as np, pandas as pd
_need = ["load_pmdata_participant", "overview", "PMDATA_DIR", "RESULTS_DIR", "ndi"]
_miss = [g for g in _need if g not in globals()]
assert not _miss, f"Run CELL 1.0, CELL 2 and CELL 5 first. Missing: {_miss}"
DIAG_DIR = os.path.join(os.path.dirname(RESULTS_DIR), "results_diagnostics"); os.makedirs(DIAG_DIR, exist_ok=True)
PID = "p12"; PDIR = os.path.join(PMDATA_DIR, PID)
P12_FEATURES = ["resting_hr", "steps", "active_minutes_total", "lightly_active_min", "moderately_active_min",
                "very_active_min", "sedentary_min", "sleep_efficiency", "sleep_minutes", "sleep_score"]   # = PMDATA_FEATURE_NAMES (CELL 6)
out = []
def say(s=""):
    print(s); out.append(str(s))

say(f"p12 folder exists: {os.path.isdir(PDIR)}")
# (a) the max-heart-rate cell, found exactly as CELL 5 finds it
hr_col = next((c for c in overview.columns if "heart" in c and "max" in c), None)
id_col = next((c for c in overview.columns if "participant" in c or "id" in c), None)
say(f"(a) participant-overview.xlsx columns used by CELL 5: id = {id_col!r}, max HR = {hr_col!r}")
say("    max HR cell for every participant (repr):")
for _, r in overview.iterrows():
    v = r[hr_col]
    say(f"      {str(r[id_col]).strip():<4} {v!r:<22} pd.isna -> {pd.isna(v)}")
row = overview[overview[id_col].astype(str).str.lower().str.strip() == PID]
cell = row.iloc[0][hr_col]
say(f"    p12 cell: repr = {cell!r}, type = {type(cell).__name__}, pd.isna = {pd.isna(cell)}")
# (b) the exact CELL 5 expression on that cell
try:
    _v = float(row.iloc[0][hr_col]) if not pd.isna(row.iloc[0][hr_col]) else np.nan
    say(f"(b) CELL 5 expression: no error (value {_v}) -> the cause is NOT the max HR cell; send the traceback in (c)")
    CAUSE_CONFIRMED = False
except Exception as e:
    msg = f"{type(e).__name__}: {e}"
    CAUSE_CONFIRMED = str(e) == "could not convert string to float: ' '"
    say(f"(b) CELL 5 expression raises {msg}")
    say(f"    identical to the CELL 5 log line 'ERROR p12: could not convert string to float: ' '': {CAUSE_CONFIRMED}")
say(f"    proposed fix pd.to_numeric(cell, errors='coerce') -> {pd.to_numeric(cell, errors='coerce')!r} (max HR is not a model feature)")

# (c) the loader on p12's folder, OUTSIDE any try block: if this line fails, the traceback below is the deliverable
say("(c) load_pmdata_participant(p12 folder), no try block:")
daily = load_pmdata_participant(PDIR)
d = pd.to_datetime(daily["date"], errors="coerce")
steps = d.diff().dt.days.dropna()
gaps = [(str(a.date()), str(b.date())) for a, b, s in zip(d.iloc[:-1], d.iloc[1:], steps) if s != 1]
say(f"    loaded: {len(daily)} rows, {d.min().date()} .. {d.max().date()}, unique dates: {d.is_unique}, "
    f"non-consecutive steps: {len(gaps)} {gaps[:10] if gaps else ''}")
say("    share missing per feature: " + ", ".join(
    f"{c} {pd.to_numeric(daily[c], errors='coerce').isna().mean():.2f}" if c in daily.columns else f"{c} ABSENT"
    for c in P12_FEATURES))

# (d) the CELL 6 / CELL 6.5 feature step on p12 alone, so a problem there shows now and not mid-run
try:
    _df = daily.copy().reset_index(drop=True)
    for c in P12_FEATURES:
        if c not in _df.columns:
            _df[c] = np.nan
        _df[c] = pd.to_numeric(_df[c], errors="coerce"); _df[c] = _df[c].fillna(_df[c].median())
    with contextlib.redirect_stdout(io.StringIO()):
        _nf = ndi.build_neutrosophic_features(_df[P12_FEATURES].values, P12_FEATURES, train_mask=np.ones(len(_df), bool))
    say(f"(d) CELL 6 feature step on p12: OK ({len(_nf)} rows, NDI mean {_nf['NDI'].mean():.4f})")
except Exception:
    say("(d) CELL 6 feature step on p12 FAILED -- send this traceback before fixing anything:")
    say(traceback.format_exc())

# (e) folds under the existing rules (the v1.4 run decides; this is the arithmetic)
n = len(daily); t0 = min(10, max(5, n // 3))
folds = ndi.blocked_tscv(np.zeros((n, 1)), np.zeros(n), ndi.rf_factory, T0=t0, g=6, return_folds=True)["folds"]
scored = sum(1 for _, te in folds if te[0] + 7 <= n - 1)   # forward target needs all 7 following rows
say(f"(e) n = {n} rows, t0 = {t0}, test folds = {len(folds)}, of which have a full 7-row forward window = {scored}")
say(f"    existing rules: >= 15 rows -> {n >= 15}; >= 10 scored test folds -> {scored >= 10}. "
    f"Expected to enter: {n >= 15 and scored >= 10} (decided by the v1.4 run, not by hand)")
say("")
say("P-1 RESULT: " + ("cause CONFIRMED as predicted (single-space max HR cell). Fix as in the memo."
                      if CAUSE_CONFIRMED else "cause NOT as predicted -- send this output to ma'am before fixing."))
with open(os.path.join(DIAG_DIR, "p12_check.txt"), "w") as f:
    f.write("\n".join(out) + "\n")
print(f"\nsaved -> {os.path.join(DIAG_DIR, 'p12_check.txt')}")

p12 folder exists: True
(a) participant-overview.xlsx columns used by CELL 5: id = 'participant_id', max HR = 'max_heart_rate'
    max HR cell for every participant (repr):
      p01  182                    pd.isna -> False
      p02  169                    pd.isna -> False
      p03  157                    pd.isna -> False
      p04  195                    pd.isna -> False
      p05  184                    pd.isna -> False
      p06  181                    pd.isna -> False
      p07  nan                    pd.isna -> True
      p08  200                    pd.isna -> False
      p09  183                    pd.isna -> False
      p10  197                    pd.isna -> False
      p11  203                    pd.isna -> False
      p12  ' '                    pd.isna -> False
      p13  186                    pd.isna -> False
      p14  190                    pd.isna -> False
      p15  nan                    pd.isna -> True
      p16  199                    pd.isna -> False
    p12 cell:

In [8]:
# P-2  p12 Fitbit files: present, size, entries (read-only)
import os, json
_fb = os.path.join(PMDATA_DIR, "p12", "fitbit")
for f in ["resting_heart_rate.json", "lightly_active_minutes.json", "moderately_active_minutes.json",
          "very_active_minutes.json", "sedentary_minutes.json", "steps.json", "sleep.json", "sleep_score.csv"]:
    p = os.path.join(_fb, f)
    if not os.path.isfile(p):
        print(f"{f:<32} MISSING"); continue
    n = "csv"
    if f.endswith(".json"):
        try:
            d = json.load(open(p)); n = f"{len(d)} entries, first: {str(d[0])[:80] if d else '-'}"
        except Exception as e:
            n = f"UNREADABLE: {type(e).__name__}: {e}"
    print(f"{f:<32} {os.path.getsize(p):>10} bytes  {n}")

resting_heart_rate.json          MISSING
lightly_active_minutes.json      MISSING
moderately_active_minutes.json         7847 bytes  152 entries, first: {'dateTime': '2019-11-01 00:00:00', 'value': '14'}
very_active_minutes.json               7852 bytes  152 entries, first: {'dateTime': '2019-11-01 00:00:00', 'value': '15'}
sedentary_minutes.json                 8203 bytes  152 entries, first: {'dateTime': '2019-11-01 00:00:00', 'value': '1169'}
steps.json                          4299815 bytes  83752 entries, first: {'dateTime': '2019-11-01 05:32:00', 'value': '19'}
sleep.json                             9948 bytes  8 entries, first: {'logId': 24564110290, 'dateOfSleep': '2019-11-07', 'startTime': '2019-11-07 14:
sleep_score.csv                         225 bytes  csv


In [5]:
# W-0  Zero counts per participant and 1-5 item, reports W3 drops, and the W1/W4 counts (v1.4 memo, Section 3; deliverable 2)
# Read-only: reads pmsys/wellness.csv directly (all participant folders, p12 included). Writes results_diagnostics/ only.
import os
import numpy as np, pandas as pd
_need = ["PMDATA_DIR", "RESULTS_DIR"]
_miss = [g for g in _need if g not in globals()]
assert not _miss, f"Run CELL 1.0 first. Missing: {_miss}"
DIAG_DIR = os.path.join(os.path.dirname(RESULTS_DIR), "results_diagnostics"); os.makedirs(DIAG_DIR, exist_ok=True)

# ---- Wellness rules W1-W4 (v1.4 memo, Section 3). Identical in NDI_v1_4_prechecks.ipynb (W-0) and NDI_v1_4.ipynb (W-1).
WELLNESS_TZ = "Europe/Oslo"
W_ITEMS_1_5 = ["mood", "stress", "fatigue", "sleep_quality", "soreness"]   # scored 1-5: 0 = missing (W2)
W_ITEMS_ALL = W_ITEMS_1_5 + ["readiness"]                                  # readiness runs 0-10: 0 is valid
W_RANGE = {**{c: (1, 5) for c in W_ITEMS_1_5}, "readiness": (0, 10)}

def load_wellness_reports(pmdata_dir, pid):
    """Every report in pmsys/wellness.csv, untouched, with its UTC and local (W1) calendar date."""
    path = os.path.join(pmdata_dir, pid, "pmsys", "wellness.csv")
    if not os.path.isfile(path):
        return None
    w = pd.read_csv(path)
    w.columns = [c.strip().lower().replace(" ", "_") for c in w.columns]
    raw = w["effective_time_frame"].astype(str).str.strip()
    ts = pd.to_datetime(raw, errors="coerce", utc=True)
    rep = pd.DataFrame({"pid": pid, "timestamp": raw.values, "ts_utc": ts.values})
    rep["ts_utc"] = pd.to_datetime(rep["ts_utc"], utc=True)
    rep["tz_aware"] = raw.str.contains(r"(?:Z|[+-]\d\d:?\d\d)$", regex=True).values
    rep["utc_date"] = rep["ts_utc"].dt.strftime("%Y-%m-%d")
    rep["local_date"] = rep["ts_utc"].dt.tz_convert(WELLNESS_TZ).dt.strftime("%Y-%m-%d")      # W1
    for c in W_ITEMS_ALL:
        rep[c] = pd.to_numeric(w[c], errors="coerce").values if c in w.columns else np.nan
    return rep

def apply_wellness_rules(rep):
    """W1-W4 for one participant. Returns (daily table: one row per local date, log dict)."""
    pid = rep["pid"].iloc[0] if len(rep) else None
    bad_ts = rep["ts_utc"].isna() | ~rep["tz_aware"]
    r = rep[~bad_ts].copy()
    log = dict(pid=pid, reports=len(rep), timestamps_unparseable_or_without_timezone=int(bad_ts.sum()),
               reports_moved_by_W1=int((r["utc_date"] != r["local_date"]).sum()))
    for c in W_ITEMS_ALL:                                   # anything other than 0 outside the scale
        lo, hi = W_RANGE[c]
        v = r[c]
        log[f"out_of_range_{c}"] = int((v.notna() & (v != 0) & ((v < lo) | (v > hi) | (v != v.round()))).sum())
    for c in W_ITEMS_1_5:
        log[f"zeros_{c}"] = int((r[c] == 0).sum())
    blank = (r[W_ITEMS_1_5] == 0).all(axis=1)              # W3: all five 1-5 items are 0 -> blank submission
    log["reports_dropped_W3"] = int(blank.sum())
    r = r[~blank].copy()
    for c in W_ITEMS_1_5:                                   # W2: 0 = missing on the 1-5 items
        log[f"zeros_set_missing_W2_{c}"] = int((r[c] == 0).sum())
        r.loc[r[c] == 0, c] = np.nan
    r = r.sort_values("ts_utc", kind="mergesort")
    g = r.groupby("local_date", sort=True)
    daily = g[W_ITEMS_ALL].last()                           # W4: last valid value of the day, item by item
    daily.insert(0, "reports_that_day", g.size())
    daily = daily.reset_index().rename(columns={"local_date": "date"})
    daily.insert(0, "pid", pid)
    log["local_days_with_a_report"] = len(daily)
    log["days_with_2plus_reports_W4"] = int((daily["reports_that_day"] >= 2).sum())
    for c in W_ITEMS_ALL:
        nvalid = g[c].count()
        nuniq = g[c].nunique()
        log[f"W4_days_choosing_{c}"] = int((nvalid >= 2).sum())         # days where W4 picked among valid values
        log[f"W4_days_values_differ_{c}"] = int((nuniq >= 2).sum())     # ... and the values disagreed
    return daily, log

def build_wellness_tables(pmdata_dir, pids):
    tables, logs = {}, []
    for pid in pids:
        rep = load_wellness_reports(pmdata_dir, pid)
        if rep is None or rep.empty:
            logs.append(dict(pid=pid, reports=0))
            continue
        tables[pid], lg = apply_wellness_rules(rep)
        logs.append(lg)
    log = pd.DataFrame(logs).fillna(0)
    for c in log.columns:
        if c != "pid":
            log[c] = log[c].astype(int)
    return tables, log

def wellness_zero_table(log):
    """Deliverable 2: zeros per participant and 1-5 item (all reports), blank reports W3 drops, zero share."""
    z = log[["pid", "reports"] + [f"zeros_{c}" for c in W_ITEMS_1_5] + ["reports_dropped_W3"]].copy()
    z["zeros_total"] = z[[f"zeros_{c}" for c in W_ITEMS_1_5]].sum(axis=1)
    z["zero_share_of_item_values"] = (z["zeros_total"] / (5 * z["reports"]).where(z["reports"] > 0)).round(3)
    z["zeros_left_after_W3"] = z["zeros_total"] - 5 * z["reports_dropped_W3"]
    return z

_pids = sorted(d for d in os.listdir(PMDATA_DIR) if os.path.isdir(os.path.join(PMDATA_DIR, d)) and d.startswith("p"))
wellness_tables, wellness_log = build_wellness_tables(PMDATA_DIR, _pids)
zeros = wellness_zero_table(wellness_log)
zeros.to_csv(os.path.join(DIAG_DIR, "wellness_zero_counts.csv"), index=False)
wellness_log.to_csv(os.path.join(DIAG_DIR, "wellness_rules_precheck_log.csv"), index=False)
pd.set_option("display.width", 250)
print("ZERO COUNTS (all reports; the five items scored 1-5), sorted by zero share")
print(zeros.sort_values("zero_share_of_item_values", ascending=False).to_string(index=False))
print(f"\nReports W3 drops (all five 1-5 items = 0): {int(zeros['reports_dropped_W3'].sum())} in total -> "
      + str({r.pid: int(r.reports_dropped_W3) for r in zeros.itertuples() if r.reports_dropped_W3}))
print(f"Zeros left after W3 (set to missing by W2): {int(zeros['zeros_left_after_W3'].sum())}")
_chk = ["timestamps_unparseable_or_without_timezone"] + [f"out_of_range_{c}" for c in W_ITEMS_ALL]
_bad = wellness_log[wellness_log[_chk].sum(axis=1) > 0]
print("\nTimestamps or values the rules cannot handle: " + ("none" if _bad.empty else "\n" + _bad[["pid"] + _chk].to_string(index=False)))
print(f"W1: reports whose local (Oslo) date differs from the UTC date: {int(wellness_log['reports_moved_by_W1'].sum())} "
      + str({r.pid: int(r.reports_moved_by_W1) for r in wellness_log.itertuples() if r.reports_moved_by_W1}))
print(f"W4: local days with 2+ reports after W3: {int(wellness_log['days_with_2plus_reports_W4'].sum())}; "
      f"mood days where W4 chose among valid values: {int(wellness_log['W4_days_choosing_mood'].sum())}, "
      f"of which the values differed: {int(wellness_log['W4_days_values_differ_mood'].sum())}")
print(f"\nsaved -> {DIAG_DIR}/wellness_zero_counts.csv, wellness_rules_precheck_log.csv")

ZERO COUNTS (all reports; the five items scored 1-5), sorted by zero share
pid  reports  zeros_mood  zeros_stress  zeros_fatigue  zeros_sleep_quality  zeros_soreness  reports_dropped_W3  zeros_total  zero_share_of_item_values  zeros_left_after_W3
p05      137           2             2              2                    2               2                   2           10                      0.015                    0
p13       72           0             1              0                    0               0                   0            1                      0.003                    1
p11      107           1             0              0                    0               0                   0            1                      0.002                    1
p04      145           0             1              0                    0               0                   0            1                      0.001                    1
p03       82           0             0              0            

In [6]:
# S-0  FYI only (not in the memo; nothing is changed): how CELL 5 dates sleep_score.csv
# CELL 5 takes the calendar date of the sleep-score timestamp as parsed. If the timestamps are UTC ("...Z"),
# a score timestamped between local midnight and 01:00 (02:00 in summer) lands on the previous date, as the
# wellness reports did. This counts how often that happens. Writes results_diagnostics/sleep_score_dating_fyi.csv.
import os
import numpy as np, pandas as pd
_rows = []
for pid in sorted(d for d in os.listdir(PMDATA_DIR) if os.path.isdir(os.path.join(PMDATA_DIR, d)) and d.startswith("p")):
    path = os.path.join(PMDATA_DIR, pid, "fitbit", "sleep_score.csv")
    if not os.path.isfile(path):
        _rows.append(dict(pid=pid, sleep_scores=0)); continue
    s = pd.read_csv(path); s.columns = [c.strip().lower() for c in s.columns]
    ts_col = next((c for c in s.columns if "timestamp" in c or "date" in c), None)   # same column CELL 5 uses
    raw = s[ts_col].astype(str).str.strip()
    tz = raw.str.contains(r"(?:Z|[+-]\d\d:?\d\d)$", regex=True)
    as_parsed = pd.to_datetime(raw, errors="coerce").dt.strftime("%Y-%m-%d")       # CELL 5 rule
    local = pd.to_datetime(raw, errors="coerce", utc=True).dt.tz_convert("Europe/Oslo").dt.strftime("%Y-%m-%d").where(tz, as_parsed)
    _rows.append(dict(pid=pid, sleep_scores=len(s), example_timestamp=raw.iloc[0] if len(raw) else "",
                      timestamps_with_timezone=int(tz.sum()), local_date_differs_from_CELL5_date=int((local != as_parsed).sum())))
ss = pd.DataFrame(_rows).fillna(0)
ss.to_csv(os.path.join(DIAG_DIR, "sleep_score_dating_fyi.csv"), index=False)
print(ss.to_string(index=False))
print(f"\nTOTAL sleep scores whose local date differs from the date CELL 5 gives them: "
      f"{int(ss['local_date_differs_from_CELL5_date'].sum())} (FYI; not changed in v1.4)")

pid  sleep_scores    example_timestamp  timestamps_with_timezone  local_date_differs_from_CELL5_date
p01           150 2019-11-01T06:29:30Z                       150                                   0
p02           138 2019-11-07T06:09:30Z                       138                                   0
p03            74 2019-11-02T07:40:00Z                        74                                   1
p04           140 2019-11-07T06:44:30Z                       140                                   0
p05           117 2019-11-01T06:48:00Z                       117                                   0
p06           147 2019-11-03T07:45:30Z                       147                                   0
p07           140 2019-11-08T07:02:30Z                       140                                   0
p08           132 2019-11-12T07:07:30Z                       132                                   0
p09           132 2019-11-01T07:28:00Z                       132                           